In [ ]:
# Task 1: Data Preparation
import spacy
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import torch
from transformers import AutoModel, AutoTokenizer
import en_core_web_sm

# Load spaCy model with word vectors
nlp = en_core_web_sm.load()

# Sample sentences
sentences = ["The sky is clear and sunny.",
             "It is a bright, sunny day.",
             "The gloomy weather includes clouds and rain."]

def get_sentence_vector(sentence, model):
    # Process the sentence and get its vector
    doc = model(sentence)
    return doc.vector

def get_sentence_similarity_matrix(sentences, model):
    # Get vectors for all sentences
    vectors = np.vstack([get_sentence_vector(sentence, model) for sentence in sentences])
    return cosine_similarity(vectors)

# Calculate similarities using spaCy's word vectors
similarities = get_sentence_similarity_matrix(sentences, nlp)

print("Sentence Similarities using spaCy (Static Embeddings):\n")
for i, sent1 in enumerate(sentences):
    for j, sent2 in enumerate(sentences):
        print(f"Similarity between:\n'{sent1}' and\n'{sent2}': {similarities[i][j]:.4f}\n")

# Load BERT model and tokenizer
model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def get_bert_sentence_vector(sentence, tokenizer, model):
    inputs = tokenizer(sentence, return_tensors="pt")
    outputs = model(**inputs)
    sentence_embedding = outputs.last_hidden_state.mean(dim=1).detach().numpy()
    return sentence_embedding

def get_bert_similarity_matrix(sentences, tokenizer, model):
    vectors = np.vstack([get_bert_sentence_vector(sentence, tokenizer, model) for sentence in sentences])
    return cosine_similarity(vectors)

# Calculate similarities using BERT
bert_similarities = get_bert_similarity_matrix(sentences, tokenizer, model)

print("\nSentence Similarities using BERT (Contextual Embeddings):\n")
for i, sent1 in enumerate(sentences):
    for j, sent2 in enumerate(sentences):
        print(f"Similarity between:\n'{sent1}' and\n'{sent2}': {bert_similarities[i][j]:.4f}\n"